# [01-3] 가중치 · 특성중요도 · Feature Engineering — EOL 최종검사 불량 선별

**K-뉴딜 아카데미 스마트팩토리 캡스톤 · 공통 미니 프로젝트 (1~2회차 · 4개 팀 동일 데이터)**

### 오늘의 목표

1. 모델의 **가중치(계수)** 와 **특성중요도**를 읽는다
2. **현장 지식으로 새 변수를 만든다** (특성공학)
3. 변수를 **줄여도 되는지** 확인한다 (특성선택 · PCA)

### 왜 해석이 필요한가

현장에서는 반드시 이렇게 묻습니다. **"그래서 뭘 고치면 되는데?"**

모델이 "이 제품은 위험합니다(확률 0.82)"라고만 말하면 아무도 움직일 수 없습니다.
**"토크가 규격을 벗어났고 야간 M03에서 만들어졌기 때문"** 이라고 말해야
점검 항목이 정해집니다.

> **주의**: 특성중요도는 "모델이 많이 참고한 변수"이지 **"현실의 원인"이 아닙니다.**
> 인과를 주장하려면 공정조건을 바꿔 보는 **실험**이 필요합니다.

---

## 1. 준비 — 라이브러리와 한글 폰트

| 도구 | 역할 |
|---|---|
| `pandas` (pd) | 표 데이터 처리 (엑셀에 해당) |
| `numpy` (np) | 숫자 계산 |
| `matplotlib` (plt) · `seaborn` (sns) | 그래프 |
| `scikit-learn` | 머신러닝 |

아래 셀은 **한 번만** 실행하면 됩니다.

In [ ]:
# [준비 셀] 내용을 자세히 이해할 필요는 없습니다. 실행만 하세요.
# 라이브러리 불러오기 + 한글 폰트 설정
import os, sys, glob, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

# 한글 폰트: 설치된 폰트를 먼저 찾고, Colab이면 koreanize-matplotlib 설치
from matplotlib import font_manager
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR"]:
    if _f in _installed:
        mpl.rcParams["font.family"] = _f
        break
else:
    if IN_COLAB:
        os.system(f"{sys.executable} -m pip install -q koreanize-matplotlib")
        import koreanize_matplotlib  # noqa

mpl.rcParams["axes.unicode_minus"] = False     # 음수 기호 깨짐 방지
mpl.rcParams["figure.figsize"] = (9, 4.5)
mpl.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("준비 완료 | pandas", pd.__version__, "| Colab:", IN_COLAB)

---

## 2. 데이터 불러오기

Colab은 구글의 임시 컴퓨터에서 돌아가므로 **데이터 파일을 올려 줘야** 합니다.
아래 셀이 흔한 경로를 먼저 찾아보고, 못 찾으면 **업로드 창**을 띄웁니다.
(드라이브를 쓰려면 맨 위 2줄의 `#` 을 지우세요.)

- 사용할 파일: `K-뉴딜_아카데미_1회차_공통MiniCase_미니프로젝트_데이터셋_수정.xlsx`

In [ ]:
# [데이터 적재 셀] 파일을 찾아서 읽어 오는 코드입니다. 실행만 하세요.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_FILE = "K-뉴딜_아카데미_1회차_공통MiniCase_미니프로젝트_데이터셋_수정.xlsx"
SEARCH_DIRS = [os.environ.get("KNEWDEAL_DATA_DIR", ""), ".", "..", "../..", "/content",
               "/content/drive/MyDrive", "/content/drive/MyDrive/K뉴딜"]

DATA_PATH = None
for d in SEARCH_DIRS:                      # ① 정확한 파일명으로 찾기
    if d and os.path.exists(os.path.join(d, DATA_FILE)):
        DATA_PATH = os.path.join(d, DATA_FILE)
        break
if DATA_PATH is None:                      # ② 키워드로 찾기
    for d in SEARCH_DIRS:
        hits = glob.glob(os.path.join(d, "*MiniCase*")) if d else []
        if hits:
            DATA_PATH = sorted(hits)[0]
            break
if DATA_PATH is None and IN_COLAB:         # ③ 직접 업로드
    from google.colab import files
    DATA_PATH = list(files.upload().keys())[0]

print("데이터 경로:", DATA_PATH)
df_raw = pd.read_excel(DATA_PATH, sheet_name="MiniCase_Data")
print("크기(행, 열):", df_raw.shape)
df_raw.head(3)

In [ ]:
# 토크 결측 복원에 쓸 '센서로그' 시트도 함께 읽는다
sensor_log = pd.read_excel(DATA_PATH, sheet_name="센서로그")
print("센서로그:", sensor_log.shape)
sensor_log.head(3)

---

## 3. 2편 내용 한 번에 재현

해석에 집중하기 위해, 2편에서 한 일(정제 → 특성 → 분리 → 학습)을 한 셀로 실행합니다.

In [ ]:
# [복습 셀] 2편에서 한 일을 그대로 모았습니다. 지금 다시 읽지 않아도 됩니다.
def clean_data(d, sensor_log):
    """꼭 필요한 정제만 수행한다."""
    d = d.copy()

    # ① 토크 결측 → 센서로그의 복원값으로 채운다 (record_id로 연결)
    restore = sensor_log.set_index("record_id")["torque_nm_restored"]
    d["torque_nm"] = d["torque_nm"].fillna(d["record_id"].map(restore))

    # ② 진동 결측 → 같은 설비의 중앙값으로 채운다
    d["vibration_mm_s"] = d["vibration_mm_s"].fillna(
        d.groupby("equipment_id")["vibration_mm_s"].transform("median"))

    # ③ defect_type(불량유형)은 구조적 결측이므로 그대로 둔다
    return d


df = clean_data(df_raw, sensor_log)
print("정제 후 남은 결측:")
print(df.isna().sum()[lambda s: s > 0].to_string())

SEED = 42
df["target"] = (df['inspection_result'] == '불합격').astype(int)
CAT = ["shift", "line_id", "product_model", "equipment_id", "operator_group"]
NUM = [
    "torque_nm", "pressure_bar", "cycle_time_sec", "temperature_c", "vibration_mm_s",
    "downtime_min", "energy_kwh"
]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore",
                                               sparse_output=False))]), CAT),
])

X_train, X_test, y_train, y_test = train_test_split(
    df[CAT + NUM], df["target"], test_size=0.25, random_state=SEED, stratify=df["target"])

logit = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
                  ]).fit(X_train, y_train)
forest = Pipeline([("pre", pre),
                   ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                                  min_samples_leaf=5,
                                                  class_weight="balanced",
                                                  random_state=SEED))]).fit(X_train, y_train)


def score(model, name, X_te=X_test, y_te=y_test, th=0.3):
    """성능 요약 한 줄 (임계값 0.3 기준)"""
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= th).astype(int)
    return {"모델": name, "AUC": roc_auc_score(y_te, proba),
            "Recall": recall_score(y_te, pred, zero_division=0),
            "Precision": precision_score(y_te, pred, zero_division=0),
            "F1": f1_score(y_te, pred, zero_division=0), "변수수": X_te.shape[1]}


display(pd.DataFrame([score(logit, "로지스틱회귀"), score(forest, "랜덤포레스트")])
        .set_index("모델").round(3))

---

## 4. 로지스틱 회귀의 가중치(계수) 읽기

로지스틱 회귀는 이렇게 계산합니다.

```
z = w₁·토크 + w₂·온도 + w₃·진동 + ...  →  확률 = 1 / (1 + e^(−z))
```

**w가 가중치(계수)** 이고, 읽는 법은 두 가지입니다.

- **부호**: 양수(+)면 불량 확률을 **높이는** 변수, 음수(−)면 **낮추는** 변수
- **크기**: 우리는 모든 변수를 **표준화**했으므로 **크기를 서로 비교할 수 있습니다**
  (표준화하지 않으면 단위가 큰 변수의 계수가 무조건 작게 나옵니다)

`exp(계수)` = **오즈비**. 2.0이면 "그 변수가 1 표준편차 커질 때 불량 가능성이 2배".

> **함정**: 서로 비슷한 변수가 있으면 계수를 **나눠 갖습니다.**
> 이 데이터의 `line_id` 와 `product_model` 은 완전히 같은 정보라서
> 계수가 쪼개지거나 부호가 뒤집힐 수 있습니다. "계수가 작다 = 안 중요하다"가 아닙니다.

In [ ]:
names = [n.replace("num__", "").replace("cat__", "")
         for n in logit.named_steps["pre"].get_feature_names_out()]
coefs = pd.DataFrame({"변수": names, "계수": logit.named_steps["clf"].coef_[0]})
coefs["오즈비"] = np.exp(coefs["계수"]).round(2)
coefs = coefs.reindex(coefs["계수"].abs().sort_values(ascending=False).index)

display(coefs.head(10).round(3))

top = coefs.head(12)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top["변수"], top["계수"],
         color=["#e15759" if v > 0 else "#4e79a7" for v in top["계수"]])
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("표준화 계수 (양수 = 불량 확률을 높임)")
plt.title("로지스틱 회귀 계수")
plt.show()

---

## 5. 랜덤포레스트 특성중요도

트리는 "이 변수로 나눴을 때 불량/합격이 얼마나 깔끔하게 갈리는가"를 기준으로 분기합니다.
그 기여도를 모두 합한 것이 **특성중요도**이며, 합이 1이 되도록 표시됩니다.

**계수와 순위가 다를 수 있습니다.** 두 방법이 서로 다른 질문에 답하기 때문입니다.

- 계수: "다른 변수가 같을 때, **직선적으로** 얼마나 기여하나"
- 특성중요도: "트리가 나눌 때 **얼마나 많이 사용했나**" (조합·비선형도 반영)

**둘 다에서 상위인 변수**가 진짜 핵심 변수입니다. 보고서에는 그 변수를 씁니다.

In [ ]:
imp = pd.DataFrame({"변수": names,
                    "중요도": forest.named_steps["clf"].feature_importances_})
imp = imp.sort_values("중요도", ascending=False)
display(imp.head(10).round(4))

top = imp.head(12)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top["변수"], top["중요도"], color="#59a14f")
plt.xlabel("특성중요도 (합 = 1)")
plt.title("랜덤포레스트 특성중요도")
plt.show()

print("상위 5개 변수가 전체 중요도의",
      f"{imp.head(5)['중요도'].sum()*100:.0f}% 를 차지합니다.")

---

## 6. Feature Engineering — 현장 지식으로 새 변수 만들기

**특성공학(feature engineering)** 은 **현장 지식을 숫자로 바꾸는 일**입니다.
실무에서 성능을 가장 크게 올리는 작업이고, 파이썬 실력보다 **공정 이해**가 중요합니다.

이 데이터에서 만들 변수 3개:

| 변수 | 계산식 | 왜 |
|---|---|---|
| `torque_out_of_spec` | 토크 < 27 또는 > 33 이면 1 | 현장 관리규격을 모델에 알려 준다 |
| `torque_center_dist` | \|토크 − 30\| | 규격 중심에서 벗어난 **정도** |
| `night_m03` | 야간 **그리고** M03 이면 1 | 1편에서 찾은 위험조건 |

`night_m03` 이 중요합니다. 선형모델은 "야간이면 +a, M03이면 +b"처럼 **따로** 더할 뿐이라,
**"둘이 겹칠 때만 위험하다"** 를 스스로 표현하지 못합니다. 사람이 만들어 줘야 합니다.

In [ ]:
def add_features(d):
    """현장 지식으로 새 변수 3개를 만든다."""
    d = d.copy()
    # ① 공정규격(27~33 N·m) 이탈 여부
    d["torque_out_of_spec"] = (((d["torque_nm"] < 27) | (d["torque_nm"] > 33))
                               .astype(int))
    # ② 규격 중심(30)에서 벗어난 정도
    d["torque_center_dist"] = (d["torque_nm"] - 30).abs()
    # ③ 1편에서 찾은 위험조건: 야간 AND M03
    d["night_m03"] = ((d["shift"] == "야간") & (d["equipment_id"] == "M03")).astype(int)
    return d

In [ ]:
df_fe = add_features(df)
NEW = ["torque_out_of_spec", "torque_center_dist", "night_m03"]
NUM_FE = NUM + NEW

X2 = df_fe[CAT + NUM_FE]
pre2 = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_FE),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore",
                                               sparse_output=False))]), CAT),
])
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, df_fe["target"], test_size=0.25, random_state=SEED, stratify=df_fe["target"])

logit2 = Pipeline([("pre", pre2),
                   ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
                   ]).fit(X2_train, y2_train)
forest2 = Pipeline([("pre", pre2),
                    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   min_samples_leaf=5,
                                                   class_weight="balanced",
                                                   random_state=SEED))]).fit(X2_train, y2_train)

display(pd.DataFrame([
    score(logit, "로지스틱 · 기존"),
    score(logit2, "로지스틱 · 파생변수 추가", X2_test, y2_test),
    score(forest, "랜덤포레스트 · 기존"),
    score(forest2, "랜덤포레스트 · 파생변수 추가", X2_test, y2_test),
]).set_index("모델").round(3))

print("성능이 거의 그대로여도 실패가 아니다. 파생변수가 '설명'을 쉽게 만들었다면 가치가 있다.")
print("보통 선형모델(로지스틱)에서 개선 폭이 더 크다. 트리는 조합을 스스로 찾기 때문이다.")

---

## 7. 특성선택 — 변수를 줄여도 될까

변수를 줄이는 이유는 성능 때문만이 아닙니다.

- 센서를 20개 상시 감시하려면 **설치·보전 비용**이 든다
- 점검표가 20줄이면 **아무도 안 본다**
- 변수가 적을수록 **설명하기 쉽다**

`SelectKBest` 는 변수 하나씩 목표변수와의 관련성을 측정해 상위 k개를 고릅니다.
간단하지만 한계도 있습니다 — **"혼자서는 약하지만 다른 변수와 함께면 강한 변수"** 를 놓칩니다.

In [ ]:
Xt = pre2.fit_transform(X2_train)
names2 = [n.replace("num__", "").replace("cat__", "")
          for n in pre2.get_feature_names_out()]

f_scores, _ = f_classif(Xt, y2_train)
uni = pd.DataFrame({"변수": names2, "F점수": f_scores}).sort_values("F점수", ascending=False)
display(uni.head(10).round(2))

# 변수 개수를 바꿔 가며 성능 확인
rows = []
for k in [3, 5, 8, 12, Xt.shape[1]]:
    if k > Xt.shape[1]:
        continue
    pipe = Pipeline([("pre", pre2), ("select", SelectKBest(f_classif, k=k)),
                     ("clf", RandomForestClassifier(n_estimators=200, max_depth=8,
                                                    min_samples_leaf=5,
                                                    class_weight="balanced",
                                                    random_state=SEED))]).fit(X2_train, y2_train)
    proba = pipe.predict_proba(X2_test)[:, 1]
    rows.append({"변수 수": k, "AUC": roc_auc_score(y2_test, proba),
                 "Recall": recall_score(y2_test, (proba >= 0.3).astype(int), zero_division=0)})
sel = pd.DataFrame(rows)
display(sel.round(3))

plt.figure(figsize=(7, 4))
plt.plot(sel["변수 수"], sel["AUC"], "o-", label="AUC")
plt.plot(sel["변수 수"], sel["Recall"], "s-", label="Recall")
plt.xlabel("사용한 변수 개수"); plt.legend()
plt.title("변수를 줄이면 성능이 얼마나 떨어지는가")
plt.show()
print("성능이 평평해지는 지점이 '충분한 변수 개수'다 → 센서 선정의 근거가 된다.")

---

## 8. PCA — 변수를 압축해서 그림으로 보기

변수가 많아도 서로 겹치는 정보가 많습니다. **PCA(주성분분석)** 는
**데이터가 가장 넓게 퍼진 방향**을 새 축으로 삼아 변수를 압축합니다.

- **반드시 표준화 먼저** — 단위가 큰 변수가 무조건 1등 축이 되기 때문입니다
- **주성분은 해석이 어렵습니다** — PC1은 "온도 0.4 + 진동 0.3 + ..." 처럼 섞인 축입니다

그래서 실무에서는 **2차원으로 줄여 눈으로 보는 용도**로 주로 쓰고,
최종 모델은 원래 변수로 만드는 경우가 많습니다(설명해야 하니까).

In [ ]:
X_num = df_fe[NUM_FE].fillna(df_fe[NUM_FE].median())
X_std = StandardScaler().fit_transform(X_num)        # ① 반드시 표준화

pca = PCA().fit(X_std)
cum = np.cumsum(pca.explained_variance_ratio_)
pcs = PCA(n_components=2).fit_transform(X_std)
y_all = df_fe["target"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(cum) + 1), cum * 100, "o-")
axes[0].axhline(80, color="gray", ls="--", label="80%")
axes[0].set_xlabel("주성분 개수"); axes[0].set_ylabel("누적 설명력 (%)")
axes[0].set_title("몇 개 축이면 충분한가"); axes[0].legend()

axes[1].scatter(pcs[y_all == 0, 0], pcs[y_all == 0, 1], s=14, alpha=0.3,
                c="#4e79a7", label="합격")
axes[1].scatter(pcs[y_all == 1, 0], pcs[y_all == 1, 1], s=45, c="#e15759",
                edgecolors="black", linewidths=0.4, label="불합격")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("주성분 2축으로 본 전체 데이터"); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"변수 {len(NUM_FE)}개 → 주성분 {int(np.argmax(cum >= 0.8) + 1)}개로 정보의 80%를 담을 수 있다.")
print("불합격(빨강)이 한쪽에 모여 있지 않다면, 몇 개 축만으로는 구분이 어렵다는 뜻이다.")

---

## 9. 정리

| 주제 | 핵심 |
|---|---|
| 계수(가중치) | 부호 = 방향, 크기 = 영향력(**표준화했을 때만** 비교 가능) |
| 특성중요도 | 트리가 많이 쓴 변수. 계수와 순위가 달라도 정상 |
| 특성공학 | **도메인 지식을 숫자로** — 성능 향상 효과가 가장 크다 |
| 특성선택 | 변수를 줄이면 비용↓ 설명력↑ |
| PCA | 압축·시각화에는 좋지만 **해석이 어렵다** |

### 캡스톤 제출물로 정리할 것
1. **핵심 변수 Top 3~5** (계수와 중요도 모두에서 상위인 것)
2. **만든 파생변수와 근거** ("왜 이 계산식이 현장 의미를 갖는가")
3. **제외한 변수와 이유** (데이터 누출 / 중복 정보)
4. **최종 변수 목록** — 이후 Blind Test까지 바꾸지 않습니다

> **마지막으로**: 특성중요도는 **원인이 아니라 단서**입니다.
> "M03에서 불량이 많이 관찰된다"까지가 데이터가 말해 주는 것이고,
> "M03이 불량을 만든다"는 **설비 점검으로 확인해야** 할 가설입니다.

---

## 연습문제 / 생각해 볼 질문

아래 문항은 정답을 맞히는 시험이 아니라, **발표자료에 쓸 문장을 만드는 연습**입니다.
코드를 조금씩 바꿔 보고, 결과를 한 문장으로 정리해 보세요.

1. 로지스틱 회귀 계수 상위 3개와 랜덤포레스트 특성중요도 상위 3개를 비교하세요. 두 결과가 다른 이유는?
2. `add_features()` 의 `night_m03` 을 빼고 다시 실행해 성능 변화를 확인하세요. 이 파생변수의 가치는 얼마입니까?
3. `SelectKBest(k=5)` 로 고른 5개 변수만으로 모델을 만들면 성능이 얼마나 떨어지나요? '센서 5개만 상시 감시한다면 무엇을 고를지' 제안하세요.